# Decision-boundary illustration
Both layouts use the same synthetic data, calibration split, prediction sets, axis bounds, and displayed observations. Calibration uses all 2,400 calibration observations; deterministic class-stratified subsamples are used only for the scatter display.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "conformal_inference" / "methods.py").is_file()
     and (p / "notebook" / "paper_illustrative_plot.ipynb").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from within the conformalHDC checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from conformal_inference.methods import ConformalHDC

OUTPUT_DIR = REPO_ROOT / "results" / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
})

## Data and common plotting grid

In [ ]:
def setup_triangle_geometry(separation_factor, rng):
    dim = 2
    ref_sigma = 1.0
    r_ref = ref_sigma * np.sqrt(dim)
    dist = separation_factor * (r_ref + r_ref)

    mu0 = np.array([0.0, 0.0])
    mu1 = np.array([dist, 0.0])
    mu2 = np.array([dist / 2, (np.sqrt(3) / 2) * dist])
    
    return np.stack([mu0, mu1, mu2]).astype(np.float32)

def generate_samples_3class(n_samples, prototypes, sigmas, rng):
    X_list = []
    y_list = []
    dim = prototypes.shape[1]
    
    for label, (mu, sigma) in enumerate(zip(prototypes, sigmas)):
        X_c = rng.randn(n_samples, dim) * sigma + mu
        X_list.append(X_c)
        y_list.append(np.full(n_samples, label))
        
    X = np.concatenate(X_list).astype(np.float32)
    y = np.concatenate(y_list).astype(int)
    return X, y

def generate_ood_blob(n_samples, prototypes, rng):
    center = np.mean(prototypes, axis=0)
    direction = np.array([0.0, -1.0]) 
    dist = np.linalg.norm(prototypes[2] - prototypes[0]) * 1.8
    mu_ood = center + direction * dist
    X_ood = rng.randn(n_samples, 2) * 1.0 + mu_ood 
    return X_ood.astype(np.float32)

# ==========================================
#    EXPERIMENT CONFIG
# ==========================================
SEED = 0
rng = np.random.RandomState(SEED)
SCORE_TYPES = ["inverse_quantile", "penalized", "sim", "ratio", "discount"]
SCORE_NAMES = ["Inv. quantile", "Penalized", 'Similarity', "CHDC-ratio","CHDC-discount"]
if len(SCORE_TYPES) > 5: SCORE_TYPES = SCORE_TYPES[:5]

SAMPLES_PER_CLASS = 1000
SEPARATION_CONST = 2.0 
SIGMAS = [1.0, 2, 3] 
ALPHA = 0.1
MARGINAL = False 

prototypes = setup_triangle_geometry(SEPARATION_CONST, rng)
X_full, y_full = generate_samples_3class(SAMPLES_PER_CLASS, prototypes, SIGMAS, rng)
X_ood = generate_ood_blob(100, prototypes, rng)

indices = np.arange(len(X_full))
rng.shuffle(indices)
n_total = len(X_full)
n_cal   = int(0.8 * n_total)
n_test  = int(0.2 * n_total)
idx_cal   = indices[: n_cal]
idx_test  = indices[n_cal :]
X_cal,   y_cal   = X_full[idx_cal],   y_full[idx_cal]
X_test,  y_test  = X_full[idx_test],  y_full[idx_test]

# Display sampling is independent of data generation and calibration.
DISPLAY_PER_CLASS = {0: 40, 1: 80, 2: 80}
DISPLAY_OOD = 30
display_rng = np.random.RandomState(SEED + 1)
display_indices = np.concatenate([
    display_rng.permutation(np.flatnonzero(y_test == c))[:DISPLAY_PER_CLASS[c]]
    for c in range(3)
])
display_ood_indices = display_rng.permutation(len(X_ood))[:DISPLAY_OOD]

# Use one expanded view and one grid for every method and both layouts.
x_min = min(X_test[:, 0].min(), X_ood[:, 0].min()) - 1.5
x_max = max(X_test[:, 0].max(), X_ood[:, 0].max()) + 1.5
y_min = min(X_test[:, 1].min(), X_ood[:, 1].min()) - 1.5
y_max = max(X_test[:, 1].max(), X_ood[:, 1].max()) + 3.0
resolution = 0.065
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, resolution),
    np.arange(y_min, y_max, resolution),
)
grid_points = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
# Zoom the display without changing the prediction-grid realization.
VIEW_XLIM = (x_min + 1.4, x_max - 1.2)
VIEW_YLIM = (y_min + 1.1, y_max - 2.1)


## Plotting helpers

In [ ]:
# Singleton regions remain light; overlaps have distinct, saturated colors.
REGION_COLORS = {
    frozenset():          "#FFFFFF",
    frozenset([0]):       "#F2D6DA",
    frozenset([1]):       "#CFE4F1",
    frozenset([2]):       "#DDE9B8",
    frozenset([0, 1]):    "#8E5BA6",
    frozenset([0, 2]):    "#D19A27",
    frozenset([1, 2]):    "#26948A",
    frozenset([0, 1, 2]): "#41434B",
}
DATA_COLORS = {0: "#B64B68", 1: "#226FAD", 2: "#638B25"}
REGION_CMAP = ListedColormap([
    REGION_COLORS[frozenset(c for c in range(3) if code & (1 << c))]
    for code in range(8)
])
REGION_NORM = BoundaryNorm(np.arange(-0.5, 8.5), REGION_CMAP.N)
METHOD_ORDER = ["standard", "inverse_quantile", "sim", "penalized", "ratio", "discount"]
METHOD_NAMES = {"standard": "Standard HDC", **dict(zip(SCORE_TYPES, SCORE_NAMES))}
MANUSCRIPT_WIDTH_MM = 166.0
PRINT_FONT_PT = 7.0

def plot_data_contrast(ax, dot_size=38, center_size=160, test_dot_size=50, render_scale=1):
    for label in range(3):
        rows = display_indices[y_test[display_indices] == label]
        ax.scatter(
            X_test[rows, 0], X_test[rows, 1],
            color=DATA_COLORS[label], s=test_dot_size * render_scale**2, alpha=0.9,
            edgecolors="white", linewidths=0.6 * render_scale, zorder=3,
        )
    ax.scatter(
        X_ood[display_ood_indices, 0], X_ood[display_ood_indices, 1],
        facecolors="none", edgecolors="black", marker="^",
        s=dot_size * render_scale**2, linewidths=1.2 * render_scale, alpha=0.8, zorder=3,
    )
    ax.scatter(
        prototypes[:, 0], prototypes[:, 1], c="black", marker="X",
        s=center_size * render_scale**2, edgecolor="white", linewidth=1.5 * render_scale, zorder=5,
    )
    ax.set_xlim(*VIEW_XLIM)
    ax.set_ylim(*VIEW_YLIM)
    ax.set_aspect("auto")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.2 * render_scale)

def add_common_legend(fig, ncol=6, fontsize=15, markersize=8, render_scale=1):
    entries = []
    for members in ([0], [1], [2], [0, 1], [0, 2], [1, 2], [0, 1, 2], []):
        key = frozenset(members)
        label = "Set: {" + ",".join(map(str, members)) + "}" if members else "Empty Set"
        entries.append(Patch(
            facecolor=REGION_COLORS[key],
            edgecolor="gray" if not members else "none",
            label=label, linewidth=render_scale,
        ))
    entries.extend(
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor=DATA_COLORS[c], markersize=10 * render_scale,
               markeredgewidth=render_scale, label=f"Class {c}")
        for c in range(3)
    )
    entries.append(Line2D(
        [0], [0], marker="^", color="w", markerfacecolor="none",
        markeredgecolor="black", markersize=markersize * render_scale,
        markeredgewidth=render_scale, label="OOD",
    ))
    legend = fig.legend(
        handles=entries, loc="lower center", bbox_to_anchor=(0.5, 0.01),
        ncol=ncol, fontsize=fontsize, frameon=True,
    )
    legend.get_frame().set_linewidth(0.8 * render_scale)

def plot_method_layout(nrows, ncols, filename_stem, method_order=METHOD_ORDER):
    if (nrows, ncols) == (2, 3):
        figure_width = MANUSCRIPT_WIDTH_MM / 25.4
        figsize = (figure_width, figure_width * 11 / 20)
        render_scale = figure_width / 20
        title_fontsize = PRINT_FONT_PT
        legend_columns, legend_fontsize = 6, PRINT_FONT_PT
        legend_space = 0.10
    elif (nrows, ncols) == (3, 2):
        figsize = (14, 15.5)
        render_scale, title_fontsize = 1, 18
        legend_columns, legend_fontsize = 4, 15
        legend_space = 0.10
    else:
        figsize = (7 * ncols, 4.75 * nrows)
        render_scale, title_fontsize = 1, 18
        legend_columns, legend_fontsize = min(6, 2 * ncols), 15
        legend_space = 0.14
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    for ax, method in zip(np.ravel(axes), method_order):
        ax.imshow(
            region_codes[method], cmap=REGION_CMAP, norm=REGION_NORM,
            extent=(x_min, x_max, y_min, y_max),
            origin="lower", interpolation="nearest", aspect="auto", zorder=1,
            rasterized=True,
        )
        plot_data_contrast(ax, render_scale=render_scale)
        title = "Standard HDC" if method == "standard" else f"Score: {METHOD_NAMES[method]}"
        ax.set_title(
            title, fontsize=title_fontsize, fontweight="bold",
            pad=6 * render_scale,
        )
    fig.tight_layout(pad=1.08 * render_scale, rect=[0, legend_space, 1, 0.95])
    add_common_legend(
        fig, ncol=legend_columns, fontsize=legend_fontsize,
        markersize=9, render_scale=render_scale,
    )
    for suffix in (".pdf", ".png"):
        output_path = OUTPUT_DIR / (filename_stem + suffix)
        fig.savefig(
            output_path, dpi=200, bbox_inches="tight",
            pad_inches=0.12 * render_scale, facecolor="white",
        )
        print(output_path)
    plt.show()
    plt.close(fig)
    return fig


## Prediction regions

In [ ]:
# Compute every prediction-set map once and reuse it across the two layouts.
region_codes = {}
overlap_fractions = {}
model_by_score = {}
for score_type in SCORE_TYPES:
    model = ConformalHDC(
        class_HVs=prototypes, class_labels=[0, 1, 2],
        sim_measure="euclidean", random_state=SEED, verbose=False,
    )
    model.compute_calib_scores(X_cal, y_cal, score_type=score_type)
    prediction_sets = model.set_valued_CP(grid_points, ALPHA, marginal=MARGINAL)
    codes = np.fromiter(
        (sum(1 << c for c in prediction_set) for prediction_set in prediction_sets),
        dtype=np.uint8, count=len(prediction_sets),
    ).reshape(xx.shape)
    region_codes[score_type] = codes
    model_by_score[score_type] = model
    cardinalities = np.array([bin(int(code)).count('1') for code in range(8)])
    overlap_fractions[score_type] = float(np.mean(cardinalities[codes] > 1))

nearest_class = np.argmin(
    np.linalg.norm(grid_points[:, None, :] - prototypes[None, :, :], axis=2),
    axis=1,
)
region_codes["standard"] = (1 << nearest_class).astype(np.uint8).reshape(xx.shape)
for score_type in ("sim", "discount"):
    print(f"{METHOD_NAMES[score_type]}: overlap occupies "
          f"{100 * overlap_fractions[score_type]:.2f}% of the common grid.")
assert overlap_fractions["discount"] < overlap_fractions["sim"]

## Two-by-three layout

In [ ]:
plot_method_layout(2, 3, "decision_boundary_comparison")

## Optional split-panel exports

In [ ]:
# Optional exports used by the earlier split-panel illustrations.
EXPORT_LEGACY_PANELS = False
if EXPORT_LEGACY_PANELS:
    plot_method_layout(
        2, 1, "decision_boundary_part1_standard_discount",
        method_order=["standard", "discount"],
    )
    plot_method_layout(
        2, 2, "decision_boundary_part2_variants",
        method_order=["sim", "ratio", "inverse_quantile", "penalized"],
    )